In [1]:
import sys
sys.path.append('..')
from src.bq_client import run_query
import pandas as pd

In [ ]:
# Cohort Retention Matrix Query
with open('../src/queries/cohort_retention.sql', 'r') as f:
    cohort_sql = f.read()

df_cohort = run_query(cohort_sql)
print(f"Cohort data loaded! Shape: {df_cohort.shape}")


# MoM Revenue Growth Query
with open('../src/queries/monthly_revenue.sql', 'r') as f:
    revenue_sql = f.read()

print("Running monthly_revenue.sql against BigQuery...")
df_revenue = run_query(revenue_sql)
print(f"Revenue data loaded! Shape: {df_revenue.shape}")

BadRequest: 400 SELECT list expression references column created_at which is neither grouped nor aggregated at [13:25]; reason: invalidQuery, location: query, message: SELECT list expression references column created_at which is neither grouped nor aggregated at [13:25]

Location: US
Job ID: 97e1be48-7b90-4780-a26e-9a6283c4840e


In [ ]:
print(" COHORT RETENTION MATRIX DATA FRAME ")
display(df_cohort.head())

In [ ]:

print("\n MONTH-OVER-MONTH REVENUE DATA FRAME")
display(df_revenue.head())

- Running Validation Tests on Cohort Matrix

In [ ]:
assert df_cohort['cohort_month'].is_unique, "FAIL: Duplicate cohort months found!"
print("Check 1 Passed: Cohort months are completely unique.")

violating_cohorts = df_cohort[df_cohort['m1'] > df_cohort['m0']]
assert len(violating_cohorts) == 0, f"FAIL: Found anomalous cohorts where m1 > m0: {violating_cohorts['cohort_month'].tolist()}"
print("Check 2 Passed: Cohort decay constraints are statistically healthy.")

- Running Validation Tests on Month-Over-Month Revenue Data


In [ ]:
assert df_revenue['month'].is_monotonic_increasing, "FAIL: Revenue timeline is not sorted chronologically!"
print("Check 1 Passed: Revenue timeline is strictly chronological.")

first_row_growth = df_revenue.iloc[0]['mom_growth_pct']
assert pd.isna(first_row_growth) or first_row_growth is None, f"FAIL: First row growth rate should be Null, got {first_row_growth}"
print("Check 2 Passed: Boundary rows are safely handling division-by-zero boundaries.")
